In [1]:
import librosa
from os import path

# --- per-dataset config (this notebook is under notebooks/thinkpad-2/) ---
DATASET = "thinkpad-2"
FEATURES_NAME = "thinkpad-2.npz"

SOURCE_PATH = path.normpath(f"../../datasets/{DATASET}")
FEATURES_OUT = path.normpath(f"../../features/{FEATURES_NAME}")
# Sidecar checkpoint while a run is in progress (deleted after successful final save)
CHECKPOINT_PATH = FEATURES_OUT + ".ckpt.npz"

# Set True only when you intentionally want to recompute from scratch
FORCE_REEXTRACT = False

# Least audio duration
AUDIO_MIN_DURATION = 2  # seconds
AUDIO_SAMPLE_RATE = 48_000

# How many frames to skip in each samples
CQT_HOP_LENGTH = 512

# Bin numbers based on octaves for input
CQT_OCTAVES = 6
CQT_BINS_PER_OCTAVE = 36

# Frame size for input uses the least frames
CQT_FEATURE_FRAMES = round(AUDIO_MIN_DURATION * AUDIO_SAMPLE_RATE / CQT_HOP_LENGTH)

# Starts from note
CQT_FMIN = librosa.note_to_hz("C1")

print({
    "dataset": DATASET,
    "source": SOURCE_PATH,
    "features_out": FEATURES_OUT,
    "checkpoint": CHECKPOINT_PATH,
    "force_reextract": FORCE_REEXTRACT,
    "audio min duration": AUDIO_MIN_DURATION,
    "feature bins": CQT_BINS_PER_OCTAVE * CQT_OCTAVES,
    "feature frames": CQT_FEATURE_FRAMES,
    "feature start Hz": CQT_FMIN,
})


{'dataset': 'thinkpad-2', 'source': '../../datasets/thinkpad-2', 'features_out': '../../features/thinkpad-2.npz', 'checkpoint': '../../features/thinkpad-2.npz.ckpt.npz', 'force_reextract': True, 'audio min duration': 2, 'feature bins': 216, 'feature frames': 188, 'feature start Hz': 32.70319566257483}


In [2]:
from os import listdir, makedirs, path, remove
from tqdm.notebook import tqdm
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def preprocess(cqt):
    """Truncates frames and zero pads features to frame length"""
    if cqt.shape[1] < CQT_FEATURE_FRAMES:
        pad_width = CQT_FEATURE_FRAMES - cqt.shape[1]
        cqt = np.pad(cqt, pad_width=((0, 0), (0, pad_width)), mode="constant")
    else:
        cqt = cqt[:, :CQT_FEATURE_FRAMES]
    return cqt


def extract_cqt_features(audio_path):
    """Extracts CQT features from an audio file."""
    try:
        y, sr = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE)
        cqt = librosa.cqt(
            y=y,
            sr=sr,
            fmin=CQT_FMIN,
            n_bins=CQT_BINS_PER_OCTAVE * CQT_OCTAVES,
            bins_per_octave=CQT_BINS_PER_OCTAVE,
            hop_length=CQT_HOP_LENGTH,
            window="hann",
        )
        cqt = np.abs(cqt)
        cqt = librosa.amplitude_to_db(cqt, ref=np.max)
        return preprocess(cqt)
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None


def save_checkpoint(features, labels, files, ckpt_path):
    np.savez_compressed(
        ckpt_path,
        features=np.asarray(features),
        labels=np.asarray(labels),
        files=np.asarray(files),
    )


makedirs(path.dirname(FEATURES_OUT) or ".", exist_ok=True)

# --- Skip full recompute when final cache already exists ---
if path.isfile(FEATURES_OUT) and not FORCE_REEXTRACT:
    data = np.load(FEATURES_OUT, allow_pickle=True)
    print(
        f"Skip extraction — cache already at {FEATURES_OUT}\n"
        f"  features={data['features'].shape}  labels={data['labels'].shape}  "
        f"unique_classes={len(np.unique(data['labels']))}"
    )
else:
    features = []
    labels = []
    files = []  # relative keys class/filename for resume
    done = set()

    # Resume mid-run checkpoint if present
    if path.isfile(CHECKPOINT_PATH) and not FORCE_REEXTRACT:
        ckpt = np.load(CHECKPOINT_PATH, allow_pickle=True)
        features = list(ckpt["features"])
        labels = list(ckpt["labels"])
        files = list(ckpt["files"].astype(str))
        done = set(files)
        print(f"Resumed checkpoint {CHECKPOINT_PATH} with {len(done)} files")
    elif FORCE_REEXTRACT and path.isfile(CHECKPOINT_PATH):
        remove(CHECKPOINT_PATH)
        print(f"FORCE_REEXTRACT: removed old checkpoint {CHECKPOINT_PATH}")

    class_names = sorted(
        n for n in listdir(SOURCE_PATH) if path.isdir(path.join(SOURCE_PATH, n))
    )

    for class_name in tqdm(class_names, desc=f"Processing {DATASET} classes"):
        class_path = path.join(SOURCE_PATH, class_name)
        audio_files = sorted(
            f for f in listdir(class_path) if path.isfile(path.join(class_path, f))
        )
        class_new = 0
        for audio_file in tqdm(audio_files, desc=f"  {class_name}", leave=False):
            key = f"{class_name}/{audio_file}"
            if key in done:
                continue
            audio_path = path.join(class_path, audio_file)
            cqt = extract_cqt_features(audio_path)
            if cqt is None:
                continue
            features.append(cqt)
            labels.append(class_name)
            files.append(key)
            done.add(key)
            class_new += 1

        # Checkpoint after each class so long runs are interrupt-safe
        if class_new or not path.isfile(CHECKPOINT_PATH):
            save_checkpoint(features, labels, files, CHECKPOINT_PATH)

    if features:
        features_arr = np.asarray(features)
        labels_arr = np.asarray(labels)
        files_arr = np.asarray(files)
        np.savez_compressed(
            FEATURES_OUT,
            features=features_arr,
            labels=labels_arr,
            files=files_arr,
        )
        print(
            f"Saved {len(features_arr)} samples -> {FEATURES_OUT}  "
            f"shape={features_arr.shape}"
        )
        if path.isfile(CHECKPOINT_PATH):
            remove(CHECKPOINT_PATH)
            print(f"Removed checkpoint {CHECKPOINT_PATH}")
    else:
        print(f"No features extracted from {SOURCE_PATH}")


Processing thinkpad-2 classes:   0%|          | 0/36 [00:00<?, ?it/s]

  A#_diminished_4:   0%|          | 0/40 [00:00<?, ?it/s]

  A#_major_4:   0%|          | 0/40 [00:00<?, ?it/s]

  A#_minor_4:   0%|          | 0/40 [00:00<?, ?it/s]

  A_diminished_4:   0%|          | 0/40 [00:00<?, ?it/s]

...

  G#_minor_4:   0%|          | 0/40 [00:00<?, ?it/s]

  G_diminished_4:   0%|          | 0/40 [00:00<?, ?it/s]

  G_major_4:   0%|          | 0/40 [00:00<?, ?it/s]

  G_minor_4:   0%|          | 0/40 [00:00<?, ?it/s]

Saved 1440 samples -> ../../features/thinkpad-2.npz  shape=(1440, 216, 188)
Removed checkpoint ../../features/thinkpad-2.npz.ckpt.npz
